# CAES mini-project: system design and simulation

Compressed Air Energy Storage (CAES) stores electricity by compressing air into a reservoir. During discharge, the air is heated and expanded through a turbine to produce electricity. The project compares how compression heat management and reservoir design affect performance.

## System variants

The six variants combine three architectures with two reservoir types:

| Architecture | Constant-volume reservoir | Constant-pressure reservoir |
|---|---|---|
| Single-stage compression and expansion; compression heat rejected | 1a | 1b |
| Single-stage with thermal energy storage (TES) | 2a | 2b |
| Two-stage compression with intercooling and two-stage expansion with reheat; no TES | 3a | 3b |

A constant-volume reservoir operates between 16 bar and a minimum pressure of 8 bar, so its pressure falls during discharge. A constant-pressure reservoir remains at 16 bar while its volume varies. Variants 2a and 2b store compression heat in a hot-water TES and reuse it to preheat discharge air before combustion. Variants 3a and 3b use equal pressure-ratio stages; intercooling heat is rejected, and fuel reheats the air before each turbine stage.

## Common specification

- Working fluid: real-gas air (CoolProp).
- Ambient conditions: 15°C and 1.013 bar.
- Maximum storage pressure: 16 bar; minimum pressure for constant-volume storage: 8 bar.
- Target output: 1 MW at the turbine/generator shaft for 4 hours (4 MWh per cycle); charge duration is also 4 hours.
- Compressor isentropic efficiency: 0.85; turbine isentropic efficiency: 0.88.
- Mechanical/generator efficiency: 0.98.
- Turbine inlet temperature: 550°C. Combustion is represented as heat addition to reach this temperature, without modelling combustion chemistry.
- Cooler and TES return pinches: 10 K. Air is cooled towards ambient temperature plus the cooler pinch before storage.
- The reservoir is treated as a uniform 0D store. Stored air is assumed to return to ambient temperature before discharge and remain near ambient during discharge.

## Performance and design tasks

Use the same two-step workflow for the selected variant:

1. **Design:** size the required components, heat exchangers, and storage volume for the common output target.
2. **Simulation:** model a complete charge/discharge cycle and evaluate performance.

The common round-trip metric values fuel at its electrical equivalent:

\[
\eta = \frac{W_{\mathrm{turbine}}}{W_{\mathrm{compressor}} + \eta_f Q_{\mathrm{combustion}}},
\qquad \eta_f = 0.40.
\]

Compare round-trip efficiency, storage volume, heat-exchanger areas, maximum temperatures, and the electrical/fuel input split. Include energy (and optionally exergy) balances, identify major losses, and quantify at least one proposed performance improvement. The final comparison should show how thermal storage, staging, and reservoir type affect these results.

Here are some sources i used to return to later (soso)

Return to Gas Turbines EENG4 S8 for compressor eq and details (soso)

Paper on which sile will the pinch temp be occuring based on the Cc and Ch (LMTD) method. https://sci-hub.fr/10.1016/B978-008042839-0/50008-2 (soso)

Also return to EENG3 S6 Lec 4 for counter flow details and eq (soso)

In [2]:
# --- imports ---
from pathlib import Path
import sys

for root in (Path.cwd(), *Path.cwd().parents):
    for package_root in (root, root / "caldis-0.3.0"):
        if (package_root / "caldis" / "__init__.py").is_file():
            sys.path.insert(0, str(package_root))
            break
    else:
        continue
    break

import numpy as np
import caldis
from CoolProp.CoolProp import PropsSI
from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Compressor
from caldis.fluids import CoolPropBackend
from caldis.components import SizingHX
from caldis.components import CombustionChamber
from caldis.components import Turbine
from caldis.solvers import solve  


Initilize paramaters

Ambient Air --> Compressor --> Cooler --> Air Storage --> Combustor --> Turbine --> Exhuast

In [3]:
# --- system definition ---
#Inlet Compressor 
T0=15 +273.15 #k
P0=1.013e5
Eta_ISC=0.85

#Cooler (water)
Tpinch=10 #k


#Air Storage
Pmax=16e5
    #Volume is Variable probably use constraint to find volume based on mass flow rate and time of storage

#Combustion Chamber
Ttit=550 +273.15 #k
Eta_Fuel=0.4

#Turbine
Eta_IST=0.85
Eta_Mech=0.98

Paramters to assume

In [9]:
# A1: First we define a mass flow rate
Mdot_air=1 #kg/s
Mdot_water=1 #kg/s

# A2: We define a guess for the water outlet temperature
T_water_out_Guess = 100 + 273.15 #K This is an assumption cuz it must be less than 400 and more than T0+Tpinch

# A3: We define the backend for the fluids which is the film convection coefficients
# The heat exchanger is a counter flow heat exchanger with water on one side and air on the other side. 
# The heat transfer coefficients are based on the type of fluid and the flow conditions. Typical values are:
#   forced convection in gas (air): 50–250 W/m²/K
#   forced convection in liquid (water): 500–10000 W/m²/K
h_film_air   = 100.0    # W/m²/K  (air side)
h_film_water = 3000.0   # W/m²/K  (water side)

T_out_comp_guess = 400 + 273.15 #K This is an assumption cuz it must be less than Ttit and more than T0

Ratio_Area_cold_to_hot = 0.5 # This is an assumption, it can be changed to see the effect on the performance of the system


In [10]:
# --- Component setup ---
be_air   = CoolPropBackend("Air")
be_water = CoolPropBackend("Water")

Source     = FluidBoundary(name="Source", p=P0, T=T0, h=None, mdot=-Mdot_air, backend=be_air)
Comp       = Compressor(name="Comp", backend=be_air, eta_is=Eta_ISC)

# Since Ch<Cc in a counter flow heat exchanger, the pinch point is on the cold side.
# As the outlet of the heat exchanger was set to be T0 + 10, 
# Then the inlet of the water on the cold side would be T0 + 10 - Tpinch= T0
# Pressure is assumed to be atm as it does not affect the heat transfer
Water_Cold = FluidBoundary(name="Water_Cold", p=P0, T=T0, h=None, mdot=-Mdot_water, backend=be_water)
Water_Hot  = FluidBoundary(name="Water_Hot",  p=None, T=None, h=None, mdot=None, backend=be_water)

# The area ratio of the heat exchanger is set to 0.5, 
# meaning that the area of the cold side is half of the area of the hot side. 
# This is a reasonable assumption for a counter flow heat exchanger.
# hfilm defined in A3
Hex = SizingHX(name="Hex", backend_hot=be_air, backend_cold=be_water, h_hot=h_film_air, h_cold=h_film_water, area_ratio_cold_to_hot=Ratio_Area_cold_to_hot)

Sink = FluidBoundary(name="Sink", p=Pmax, T=T0 + Tpinch, h=None, mdot=None, backend=be_air)

In [11]:
# --- System wiring & start points ---
sys_ = System("Comp_Parts")
sys_.add(Source, Comp, Hex, Sink, Water_Cold, Water_Hot)

sys_.connect(Source.port,     Comp.inl)

sys_.connect(Comp.out,        Hex.hot_in)

sys_.connect(Water_Cold.port, Hex.cold_in)
sys_.connect(Water_Hot.port,  Hex.cold_out)

sys_.connect(Hex.hot_out,     Sink.port)

# Start-point gives the solver a physically sensible initial guess.
# Initially when we calculated the output temp in the 1st step we got 410°C
# The compressor outlet T here isn't known yet, but ~400°C is a good guess 

sys_.set_start_port(Comp.inl,    p=P0,   T=T0,          mdot= Mdot_air)
sys_.set_start_port(Comp.out,    p=Pmax, T=T_out_comp_guess,   mdot=-Mdot_air)
sys_.set_start_port(Hex.hot_in,  p=Pmax, T=T_out_comp_guess,   mdot=Mdot_air)
sys_.set_start_port(Hex.hot_out, p=Pmax, T=T0 + Tpinch,    mdot= -Mdot_air)

# Water side
sys_.set_start_port(Hex.cold_in,  p=P0, T=T0,           mdot= Mdot_water)
sys_.set_start_port(Hex.cold_out, p=P0, T=T_water_out_Guess,   mdot= -Mdot_water)
sys_.init_starts()

sol = solve(sys_)

✓ DOF: square (32 unknowns)
✓ start point: evaluable
✓ SOLVED (|F_scaled|=5.0e-10)


In [12]:
#sys_.structure()
Comp.describe()
Hex.describe()

=== Compressor('Comp') ===
  ports (p [bar], mdot [kg/s], T [°C]):
    inl        p=1.013  mdot=+1  h=414.4kJ/kg  T=15.00
    out        p=16.000  mdot=-1  h=822.4kJ/kg  T=410.60
  outputs:
    Wdot             = -408069
    pressure_ratio   = 15.7947
=== SizingHX('Hex') ===
  variables:
    UA               = 4587.68
    A_hot            = 48.9352
  ports (p [bar], mdot [kg/s], T [°C]):
    hot_in     p=16.000  mdot=+1  h=822.4kJ/kg  T=410.60
    hot_out    p=16.000  mdot=-1  h=421.0kJ/kg  T=25.00
    cold_in    p=1.013  mdot=+1  h=63.1kJ/kg  T=15.00
    cold_out   p=1.013  mdot=-1  h=464.5kJ/kg  T=99.97
  outputs:
    U_overall_hot    = 93.75
    U_overall_cold   = 187.5
    A_cold           = 24.4676
    A_total          = 73.4029
    Qdot             = 401395
    DTLM             = 87.4942


{'variables': {'UA': [4587.679369767464], 'A_hot': [48.935246610530726]},
 'ports': {'hot_in': {'p': 1600000.0, 'mdot': 1.0, 'h': 822443.6592563742},
  'hot_out': {'p': 1600000.0, 'mdot': -1.0, 'h': 421048.5357981099},
  'cold_in': {'p': 101300.0, 'mdot': 1.0, 'h': 63076.806869169945},
  'cold_out': {'p': 101300.0, 'mdot': -1.0, 'h': 464471.93040151737}},
 'outputs': {'U_overall_hot': 93.75,
  'U_overall_cold': 187.5,
  'A_cold': 24.467623305265363,
  'A_total': 73.40286991579609,
  'Qdot': 401395.12345826434,
  'DTLM': 87.49415360025472}}